# Logic: Statements, Connectives, and Truth Tables

**This notebook teaches logic from scratch, no prior exposure is assumed.**

Logic is the study of reasoning made precise. It begins with **statements**, sentences that are either true or false, and gives us exact rules for combining them with words like *and*, *or*, *not*, and *if…then*. These rules are the foundation for defining a study cohort ("diabetic **and** over 60"), for proving mathematical facts, and even for how computer chips work. This notebook builds that language step by step and verifies each rule by direct computation.

## Learning objectives

- Recognize what is and is not a **statement** (proposition).
- Evaluate compound statements built from the connectives $\lnot,\ \land,\ \lor,\ \rightarrow,\ \leftrightarrow$.
- Build a complete **truth table** for any compound statement.
- Verify logical equivalences (De Morgan's laws, the contrapositive) by comparing truth tables.
- Rebuild every connective from **NAND** alone, the way real hardware does.
- Read the quantifiers $\forall$ and $\exists$, and negate them.
- Express a study's **inclusion criteria** as a compound statement, and negate it with De Morgan.

## Background

A **statement** (or **proposition**) is a sentence that communicates a single truth value, it is either **true** or **false**. "I washed the car" and "$x = 1$" are statements; "What time is it?" (a question) and "Come here!" (a command) are not. We name statements with letters, like $p$ and $q$, and combine them with **connectives**:

$$\lnot p \ (\text{not}), \qquad p \land q \ (\text{and}), \qquad p \lor q \ (\text{or}), \qquad p \rightarrow q \ (\text{if } p \text{ then } q), \qquad p \leftrightarrow q \ (p \text{ if and only if } q).$$

A **truth table** lists a compound statement's value for every combination of its inputs. (Boolean logic is named for **George Boole**, who first treated true/false as algebra.)

## This notebook covers

1. What is a statement?
2. The five connectives.
3. Building a full truth table.
4. De Morgan's laws.
5. NAND and functional completeness.
6. Inclusive vs exclusive "or".
7. Converse, inverse, and contrapositive.
8. Quantifiers.
9. Application: writing (and negating) a study cohort.

**Prerequisites:** none, this notebook opens the unit.

**Dataset:** none, truth values are enumerated directly.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

## 1. What is a statement?

A **statement** is a sentence with a definite truth value, it must be either true or false. Questions and commands are **not** statements, because they are neither true nor false:

| Sentence | Statement? |
|---|---|
| "I washed the car." | yes (true or false) |
| "$x = 1$" | yes (true or false, once $x$ is known) |
| "What time is it?" | no (a question) |
| "Come here!" | no (a command) |

In Python a statement is just a value that is `True` or `False`. We name statements with letters, exactly as the lecture does.

In [2]:
# Name two statements. Here p and q stand for specific sentences.
p = True    # p = "I went to the store"
q = False   # q = "I left my house"

print("p =", p)
print("q =", q)

p = True
q = False


## 2. The five connectives

Each connective combines statements into a new statement. Using $p$ = "I went to the store" and $q$ = "I left my house":

| Symbol | Name | In words | True when… |
|---|---|---|---|
| $\lnot p$ | NOT | "I did **not** go to the store" | $p$ is false |
| $p \land q$ | AND | "…store **and** …house" | **both** are true |
| $p \lor q$ | OR | "…store **or** …house" | **at least one** is true |
| $p \rightarrow q$ | IMPLIES | "**if** store **then** house" | *unless* $p$ is true and $q$ is false |
| $p \leftrightarrow q$ | IFF | "store **if and only if** house" | both sides **match** |

Two connectives trip people up. **Implication** $p \rightarrow q$ is false in exactly one case, when the "if" part is true but the "then" part is false, i.e. when the promise turns out to be **a lie**; in every other case it counts as true. **Biconditional** $p \leftrightarrow q$ is simply "$p$ equals $q$". We write one small, clearly named function for each.

In [3]:
# One small function per connective. Each returns a True/False value.

def NOT(p):
    return not p

def AND(p, q):
    return p and q

def OR(p, q):
    return p or q

def IMPLIES(p, q):
    # false only when p is True and q is False ("a lie"); true otherwise
    return (not p) or q

def IFF(p, q):
    # true when both sides are the same
    return p == q

print("NOT True           =", NOT(True))
print("True AND False     =", AND(True, False))
print("True OR False      =", OR(True, False))
print("True IMPLIES False =", IMPLIES(True, False))
print("True IFF False     =", IFF(True, False))

NOT True           = False
True AND False     = False
True OR False      = True
True IMPLIES False = False
True IFF False     = False


## 3. Building a full truth table

To see a connective's *entire* behavior, we list every combination of inputs. With two statements there are only four: (T, T), (T, F), (F, T), (F, F). We build the table with two simple loops, one over the possible values of $p$, one over the possible values of $q$, collecting one row per combination.

In [4]:
# Build every row of the truth table with two plain loops.
rows = []

for p in [True, False]:
    for q in [True, False]:
        rows.append({
            "p": p,
            "q": q,
            "not p": NOT(p),
            "p and q": AND(p, q),
            "p or q": OR(p, q),
            "p -> q": IMPLIES(p, q),
            "p <-> q": IFF(p, q),
        })

truth_table = pd.DataFrame(rows)
truth_table

,p,q,not p,p and q,p or q,p -> q,p <-> q
0,True,True,False,True,True,True,True
1,True,False,False,False,True,False,False
2,False,True,True,False,True,True,False
3,False,False,True,False,False,True,True


Read any row as one scenario. The row $p=\text{True},\ q=\text{False}$ shows $p \rightarrow q$ is **False**, the single "a lie" case where "if $p$ then $q$" fails.

## 4. De Morgan's laws

De Morgan's laws tell you how to push a "not" through an "and" or an "or":

$$\lnot(p \land q) = \lnot p \lor \lnot q, \qquad \lnot(p \lor q) = \lnot p \land \lnot q.$$

We *prove* **both** for every case by building each side and checking the columns are identical.

In [5]:
rows = []

for p in [True, False]:
    for q in [True, False]:
        # first law:  not(p and q) == (not p) or (not q)
        left1 = NOT(AND(p, q))
        right1 = OR(NOT(p), NOT(q))
        # second law: not(p or q)  == (not p) and (not q)
        left2 = NOT(OR(p, q))
        right2 = AND(NOT(p), NOT(q))
        rows.append({
            "p": p, "q": q,
            "not(p and q)": left1,
            "(not p) or (not q)": right1,
            "match?": left1 == right1,
            "not(p or q)": left2,
            "(not p) and (not q)": right2,
            "match? ": left2 == right2,
        })

pd.DataFrame(rows)

,p,q,not(p and q),(not p) or (not q),match?,not(p or q),(not p) and (not q),match?
0,True,True,False,False,True,False,False,True
1,True,False,True,True,True,False,False,True
2,False,True,True,True,True,False,False,True
3,False,False,True,True,True,True,True,True


Every entry in both **match?** columns is True, so each law holds in all four cases. Read them as a pair: pushing a negation inward **flips the connective**, $\land$ becomes $\lor$, and $\lor$ becomes $\land$. That flip is the whole content of De Morgan, and section 9 puts it to work on a real filter.

## 5. NAND and functional completeness

Define one more connective, **NAND** ("not and"): $p \text{ NAND } q = \lnot(p \land q)$. It is remarkable because **every** other connective can be rebuilt from NAND alone, a property called **functional completeness**. This is exactly how modern computer chips work: they are built almost entirely from tiny NAND gates. The identities are:

$$\lnot A = A \text{ NAND } A, \qquad A \land B = (A \text{ NAND } B)\text{ NAND }(A \text{ NAND } B), \qquad A \lor B = (A \text{ NAND } A)\text{ NAND }(B \text{ NAND } B).$$

Let's verify all three by truth table.

In [6]:
def NAND(p, q):
    return not (p and q)

rows = []
for A in [True, False]:
    for B in [True, False]:
        not_via_nand = NAND(A, A)                        # rebuild NOT A
        and_via_nand = NAND(NAND(A, B), NAND(A, B))      # rebuild A and B
        or_via_nand = NAND(NAND(A, A), NAND(B, B))       # rebuild A or B
        rows.append({
            "A": A, "B": B,
            "not A": (not A), "NAND-not": not_via_nand,
            "A and B": (A and B), "NAND-and": and_via_nand,
            "A or B": (A or B), "NAND-or": or_via_nand,
        })

pd.DataFrame(rows)

,A,B,not A,NAND-not,A and B,NAND-and,A or B,NAND-or
0,True,True,False,False,True,True,True,True
1,True,False,False,False,False,False,True,True
2,False,True,True,True,False,False,True,True
3,False,False,True,True,False,False,False,False


In every row the "NAND-…" columns match the plain `not`, `and`, and `or` columns: NAND alone can express them all. A whole processor's logic is built from this one gate.

## 6. Inclusive vs exclusive "or"

Everyday "or" is ambiguous. The logical $\lor$ is the **inclusive** or: true when **at least one** input is true (including when *both* are). Sometimes we want the **exclusive** or (**XOR**): true when **exactly one** input is true, but false when both are ("soup or salad, not both").

In [7]:
def XOR(p, q):
    return p != q      # true when the two differ, i.e. exactly one is True

rows = []
for p in [True, False]:
    for q in [True, False]:
        rows.append({"p": p, "q": q,
                     "p or q (inclusive)": p or q,
                     "p XOR q (exclusive)": XOR(p, q)})
pd.DataFrame(rows)

,p,q,p or q (inclusive),p XOR q (exclusive)
0,True,True,True,False
1,True,False,True,True
2,False,True,True,True
3,False,False,False,False


The two columns differ in only one row, when **both** are True: inclusive "or" says True, exclusive "or" says False.

## 7. Converse, inverse, and contrapositive

Starting from an implication $p \rightarrow q$, there are three related statements:

- **Converse:** $q \rightarrow p$
- **Inverse:** $\lnot p \rightarrow \lnot q$
- **Contrapositive:** $\lnot q \rightarrow \lnot p$

A statement and its **contrapositive** are always equivalent; the converse and inverse are *not* equivalent to the original. We confirm with a truth table.

In [8]:
rows = []
for p in [True, False]:
    for q in [True, False]:
        original = IMPLIES(p, q)                     # p -> q
        converse = IMPLIES(q, p)                     # q -> p
        inverse = IMPLIES(NOT(p), NOT(q))            # (not p) -> (not q)
        contrapositive = IMPLIES(NOT(q), NOT(p))     # (not q) -> (not p)
        rows.append({
            "p": p, "q": q,
            "p -> q": original, "converse": converse,
            "inverse": inverse, "contrapositive": contrapositive,
            "orig == contrapos?": original == contrapositive,
        })
pd.DataFrame(rows)

,p,q,p -> q,converse,inverse,contrapositive,orig == contrapos?
0,True,True,True,True,True,True,True
1,True,False,False,True,True,False,True
2,False,True,True,False,False,True,True
3,False,False,True,True,True,True,True


The **p -> q** and **contrapositive** columns are identical, while the converse and inverse differ from the original. This is why, to prove "if $p$ then $q$", it is often easier to prove the contrapositive "if not $q$ then not $p$".

## 8. Quantifiers

Two quantifiers let us make statements about a whole collection:

- $\forall x$ ("for all $x$"), the statement holds for **every** item. *Example:* $\forall\, x \ge 0,\ \sqrt{x}$ is defined.
- $\exists x$ ("there exists $x$"), the statement holds for **at least one** item. *Example:* $\exists\, x \in \mathbb{R}$ such that $x^2 = 2$.

Negating a quantifier **swaps** it and negates the inside:

$$\lnot(\forall x\, P(x)) = \exists x\, \lnot P(x), \qquad \lnot(\exists x\, P(x)) = \forall x\, \lnot P(x).$$

So "it is **not** true that $\sqrt{x}$ is defined for all $x$" becomes "there **exists** an $x$ for which $\sqrt{x}$ is undefined" (namely any negative $x$). In Python, "for all" is `all(...)` and "there exists" is `any(...)`; we can also build them with a plain loop.

In [9]:
# Is sqrt(x) defined for ALL x in this list?  (defined means x >= 0)
values = [0, 1, 4, 9]

for_all_defined = True          # "for all": assume true, look for a counterexample
for x in values:
    if x < 0:
        for_all_defined = False

there_exists_negative = False   # "there exists": assume false, look for one case
for x in values:
    if x < 0:
        there_exists_negative = True

print("For all x, sqrt(x) is defined? ", for_all_defined)
print("There exists a negative x?     ", there_exists_negative)

For all x, sqrt(x) is defined?  True
There exists a negative x?      False


## 9. Application: writing a study cohort

Everything so far has been $p$ and $q$ standing for sentences. Here is what the connectives are
actually *for*.

A clinical study defines who may take part with **inclusion criteria**, a compound statement that
must come out true for a patient to be enrolled. Suppose we want:

> diabetic **and** (hypertensive **or** obese), but **not** pregnant

That is exactly $d \land (h \lor o) \land \lnot g$, built from the same five connectives. Written as a
database query or an electronic-health-record filter it looks almost identical, `AND`, `OR` and
`NOT` are these connectives, spelled in capitals.

Below, eight patients with their conditions recorded as `True`/`False`. We evaluate the criteria one
patient at a time, exactly as the truth tables above evaluate one row at a time.


In [10]:
# Eight patients, one row each. (The same small cohort returns in U1-2_Sets-1_Operations.)
cohort = pd.DataFrame({
    "Patient":      ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8"],
    "Diabetes":     [True,  True,  False, True,  False, True,  False, False],
    "Hypertension": [True,  False, True,  True,  False, False, True,  False],
    "Obesity":      [False, True,  True,  False, False, True,  True,  False],
    "Pregnant":     [False, False, True,  False, True,  False, False, False],
})

# Apply the criteria to one patient at a time, using the connectives from section 2.
eligible = []
for i in range(len(cohort)):
    d = cohort.loc[i, "Diabetes"]
    h = cohort.loc[i, "Hypertension"]
    o = cohort.loc[i, "Obesity"]
    g = cohort.loc[i, "Pregnant"]

    meets = AND(AND(d, OR(h, o)), NOT(g))      # d and (h or o) and not g
    eligible.append(meets)

cohort["Eligible"] = eligible
cohort


,Patient,Diabetes,Hypertension,Obesity,Pregnant,Eligible
0,P1,True,True,False,False,True
1,P2,True,False,True,False,True
2,P3,False,True,True,True,False
3,P4,True,True,False,False,True
4,P5,False,False,False,True,False
5,P6,True,False,True,False,True
6,P7,False,True,True,False,False
7,P8,False,False,False,False,False


Read one row and check it by hand. A patient enrols only if **all three** parts hold at once: the
diabetes flag is true, at least one of hypertension/obesity is true, and the pregnancy flag is false.
Any single failure makes the whole conjunction false.

Now the payoff. Suppose the ethics board asks the opposite question, *who was excluded?* That is the
**negation** of the criteria, and De Morgan's laws from section 4 rewrite it without guesswork:

$$\lnot\big(d \land (h \lor o) \land \lnot g\big) \;=\; \lnot d \;\lor\; (\lnot h \land \lnot o) \;\lor\; g$$

In words: excluded if **not diabetic**, *or* **neither hypertensive nor obese**, *or* **pregnant**.
Notice how each $\land$ became $\lor$, each $\lor$ became $\land$, and the double negative on $g$
collapsed. Rather than trust that, we check it against the direct negation for every patient.


In [11]:
# The negated criteria two ways: pushed inward by De Morgan, and negated directly.
rows = []
for i in range(len(cohort)):
    d = cohort.loc[i, "Diabetes"]
    h = cohort.loc[i, "Hypertension"]
    o = cohort.loc[i, "Obesity"]
    g = cohort.loc[i, "Pregnant"]

    direct = NOT(AND(AND(d, OR(h, o)), NOT(g)))          # negate the whole thing
    de_morgan = OR(OR(NOT(d), AND(NOT(h), NOT(o))), g)   # pushed inward by De Morgan

    rows.append({
        "Patient": cohort.loc[i, "Patient"],
        "excluded (direct)": direct,
        "excluded (De Morgan)": de_morgan,
        "match?": direct == de_morgan,
    })

pd.DataFrame(rows)


,Patient,excluded (direct),excluded (De Morgan),match?
0,P1,False,False,True
1,P2,False,False,True
2,P3,True,True,True
3,P4,False,False,True
4,P5,True,True,True
5,P6,False,False,True
6,P7,True,True,True
7,P8,True,True,True


Every row matches, so the two forms describe exactly the same group of patients, De Morgan's laws
hold on a real compound condition, not just on the two-variable truth table of section 4.

That is worth more than it looks. Rewriting a filter into an equivalent form is something you will do
constantly: to make a query readable, to make it run faster, or to check that a colleague's version
means the same thing as yours. Logic is what tells you the rewrite is safe.


## 10. Summary

- A **statement** has a definite truth value; questions and commands do not.
- **Connectives** ($\lnot, \land, \lor, \rightarrow, \leftrightarrow$) combine statements; **truth tables** list every case. Implication is false only in the one "a lie" case.
- **De Morgan's laws** push a negation through and/or; the **contrapositive** equals the original implication.
- **NAND is functionally complete**: every connective rebuilds from it, which is how real chips compute.
- **Inclusive** vs **exclusive** "or" differ only when both inputs are true.
- **Quantifiers** $\forall$ and $\exists$ describe whole collections and negate by swapping.
- Inclusion criteria for a study are a **compound statement**; negating them to describe who was
  excluded is De Morgan's laws doing real work.

Next: the same grouping questions asked with **sets** rather than connectives
(`U1-2_Sets-1_Operations`).